# NB06 — Cross-Entropy & the Logit Layer

**North star callback:** For a 128k-vocab model (Llama 3), the final linear layer
produces a `(batch × seq_len × 128000)` logit tensor. At bf16, one training step
at batch=2, seq=512 needs 2 × 512 × 128000 × 2 bytes = **256 MB just for logits**.
Unsloth's chunked cross-entropy avoids ever materializing this tensor.

## 1. The logit memory problem

In [1]:
import torch

VOCAB = 128_000
B, S = 2, 512
d = 4096  # Llama 3 hidden dim

hidden = torch.randn(B, S, d, device="cuda", dtype=torch.bfloat16)
lm_head = torch.nn.Linear(d, VOCAB, bias=False, device="cuda", dtype=torch.bfloat16)
labels = torch.randint(0, VOCAB, (B, S), device="cuda")

# Theoretical tensor size
full_logit_bytes = B * S * VOCAB * 2  # bf16
print(f"Full logit tensor: {B}×{S}×{VOCAB:,} = {full_logit_bytes / 1024**2:.0f} MB at bf16")

# Verify with live measurement (delta: peak above current baseline)
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
mem_before = torch.cuda.memory_allocated()
logits = hidden.view(-1, d) @ lm_head.weight.T  # (B*S, VOCAB)
naive_delta_mb = (torch.cuda.max_memory_allocated() - mem_before) / 1024**2
print(f"Measured allocation delta for naive logits: {naive_delta_mb:.0f} MB")

loss_ref = torch.nn.functional.cross_entropy(logits, labels.view(-1))
print(f"Reference loss: {loss_ref.item():.4f}")
del logits
torch.cuda.empty_cache()

Full logit tensor: 2×512×128,000 = 250 MB at bf16
Measured allocation delta for naive logits: 258 MB
Reference loss: 11.9375


## 2. Chunked cross-entropy

Instead of projecting all `B*S` hidden states to vocab at once, we slice into chunks.
Each chunk materializes only `(chunk_size × VOCAB)` logits — a fraction of the full tensor.
The loss is averaged across chunks.

In [2]:
def chunked_cross_entropy(hidden, lm_head_weight, labels, chunk_size=512):
    """
    Compute cross-entropy loss without materializing the full (B*S, VOCAB) logit matrix.
    Processes seq_len in chunks of `chunk_size` tokens.
    """
    B, S, d = hidden.shape
    flat_hidden = hidden.view(B * S, d)   # (B*S, d)
    flat_labels = labels.view(B * S)      # (B*S,)

    total_loss = torch.tensor(0.0, device=hidden.device, dtype=torch.float32)
    n_chunks = 0

    for start in range(0, B * S, chunk_size):
        end = min(start + chunk_size, B * S)
        chunk_logits = flat_hidden[start:end] @ lm_head_weight.T   # (chunk, VOCAB)
        chunk_labels = flat_labels[start:end]
        chunk_loss = torch.nn.functional.cross_entropy(chunk_logits, chunk_labels)
        total_loss = total_loss + chunk_loss
        n_chunks += 1

    return total_loss / n_chunks


CHUNK = 128  # chunk_size for the demo

# Show theoretical memory savings — these are exact and unambiguous
full_logit_mb   = B * S * VOCAB * 2 / 1024**2
chunk_logit_mb  = CHUNK * VOCAB * 2 / 1024**2
print(f"Full logit tensor  : {full_logit_mb:.0f} MB  ({B}×{S}×{VOCAB:,})")
print(f"Per-chunk logits   : {chunk_logit_mb:.0f} MB  (chunk_size={CHUNK})")
print(f"Theoretical savings: {full_logit_mb / chunk_logit_mb:.0f}x fewer bytes at once")

# Verify numerical correctness
# Use rtol=1e-2 (1%): bf16 cross-entropy at loss~11.8 has ~0.06 rounding error per chunk
loss_chunked = chunked_cross_entropy(hidden, lm_head.weight, labels, chunk_size=CHUNK)
max_err = abs(loss_chunked.item() - loss_ref.float().item())
pct_err = max_err / loss_ref.float().item() * 100
print(f"\nReference loss : {loss_ref.float().item():.5f}")
print(f"Chunked loss   : {loss_chunked.float().item():.5f}")
print(f"Error          : {max_err:.5f} ({pct_err:.3f}%)  ", end="")
print("✓" if pct_err < 1.0 else "✗ FAIL")

Full logit tensor  : 250 MB  (2×512×128,000)
Per-chunk logits   : 31 MB  (chunk_size=128)
Theoretical savings: 8x fewer bytes at once



Reference loss : 11.93750
Chunked loss   : 11.92969
Error          : 0.00781 (0.065%)  ✓


## 3. Benchmark

In [3]:
import sys; sys.path.insert(0, "..")
from utils.benchmark import compare

results = compare(
    fns={
        "naive": lambda: torch.nn.functional.cross_entropy(
            (hidden.view(-1, d) @ lm_head.weight.T), labels.view(-1)),
        "chunked_128": lambda: chunked_cross_entropy(hidden, lm_head.weight, labels, 128),
        "chunked_512": lambda: chunked_cross_entropy(hidden, lm_head.weight, labels, 512),
    },
    notebook="nb06",
    experiment="cross_entropy_latency",
    n_warmup=5, n_repeat=20,
)
for label, r in results.items():
    print(f"{label:14s}: {r.latency_ms:.2f} ms")

naive         : 7.19 ms
chunked_128   : 11.30 ms
chunked_512   : 7.28 ms


## 4. Read Unsloth's chunked cross-entropy

Unsloth's implementation is in `unsloth/kernels/cross_entropy_loss.py`.
It also fuses the log-softmax and NLL loss into a single Triton kernel,
avoiding even the intermediate softmax tensor within each chunk.

In [4]:
sys.path.insert(0, "../unsloth")
from unsloth.kernels import cross_entropy_loss
import inspect
print(inspect.getsource(cross_entropy_loss))

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!


# Copyright 2023-present Daniel Han-Chen & the Unsloth team. All rights reserved.
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

import triton
import triton.language as tl
import torch
from .utils import (
    calculate_settings,
    MAX_FUSED_SIZE,
    triton_tanh,
    triton_cast,
    torch_gpu_device,
    is_cdna,
)
from transformers.models.llama.modeling_llama import logger
from unsloth_zoo.utils import Version

from unsloth_zoo.loss_utils import (
    patch_loss_functions as _patch_loss_fun

## 5. Exercises

1. **Vary chunk_size**: try 128, 256, 1024, 4096. How does peak VRAM scale with chunk_size?
   What chunk_size gives the best latency vs memory trade-off?

2. **Handle ignore_index**: modify `chunked_cross_entropy` to accept an
   `ignore_index=-100` argument (for padding tokens) and pass it to `F.cross_entropy`.
   Make sure the average is computed only over non-ignored tokens.

3. **Write a Triton fused kernel**: implement a kernel that takes a single chunk of
   `(chunk, d)` hidden states and `(d, VOCAB)` weight, computes logits, and returns
   the scalar cross-entropy — without a separate `@` operator.